Code was completed using Claude. 
Paths were changed from ./nd_data/etc.. to data/nd_data/etc..


---

## **Part 1 — Exploring Different Modalities / Representations of Network Traffic**

Synthetic network traffic generation is useful for many applications such as dataset augmentation, network testing, and resource management. Many existing generation methods treat traffic generation as either a time-series prediction task or an autoregressive modeling task. In these approaches, models are trained directly on structured representations of packets—one common example is **nPrint**, a tabular format that encodes packet header fields from PCAP traces into machine-learning-friendly numerical vectors.

In an **nPrint**, each row represents one packet in a trace. The entire nPrint file (a large CSV) represents all packets of that trace in sequential order. Header fields are encoded using one-hot–like binary indicators (e.g., `1`, `0`, or `01`), making them easy to feed into ML models.

However, general-purpose ML models—even advanced time-series architectures and transformers—often struggle to capture the *complex dependencies* present in real network traffic:

* **Local dependencies**: relationships among columns within a single row (i.e., dependencies among header fields of a single packet).
* **Global dependencies**: relationships across rows (i.e., the evolution of packets within the same flow).
  For example: if the first packet of a flow uses TCP, the second packet in that same flow should also be TCP; sequence numbers, flags, and flow identifiers evolve in structured ways over time.

While sequential ML models struggle with these multi-scale dependencies, **vision models** (e.g., diffusion models) excel at capturing both local and global structure when data is presented spatially—like an image. By converting nPrint traces into 2D PNG images, we can take advantage of the strong representational capabilities of image models and generate synthetic traffic using visual generative approaches.

---

### **Your Task for Part 1**

In this part of the assignment, you will:

1. **Inspect the raw nPrint files** (found in the `real_nprints` directory).
2. **Understand how each CSV row and column corresponds to packet-level metadata.**
3. **Follow the provided conversion pipeline** that transforms these nPrint CSVs into PNG image representations suitable for use with diffusion models and other visual architectures.
4. **Take an already generated set of image-representation of images and convert them back into nprint representation for downstream task utilization**

This will help you understand why converting network traces to images can unlock generative modeling capabilities that traditional ML approaches struggle with.

Q1:
First, download and unzip the data you will need from (https://drive.google.com/file/d/1hY6nNXEYOwl1l-O_nCknO9xezcHr6ZXi/view?usp=sharing)
In your own words, describe how an nPrint CSV encodes a network trace.
Why might a 2D image representation capture structural relationships that a row-by-row CSV cannot?

**Answer**

A nPrint CSV encodes a network trace by making each row of the trace into a packet and each column into a bit that is either -1,0, or 1. 

A 2D image representation keeps the locality and context for the X and Y dimensions at once. A row-by-row CSV flattens data into a one dimension stream, getting rid of the physical connection between data points.

Q2. Design a method for converting nPrint representations of traces (in the folder real_nprints) into image representations.
Your image representation should use only the first 1024 packets from each trace to avoid producing images that are too large.
Save the images into a folder called './nd_data/student_converted_images'

In [5]:
import os
import numpy as np
import pandas as pd
from PIL import Image

INPUT_DIR = 'data/nd_data/real_nprints'
OUTPUT_DIR = 'data/nd_data/student_converted_images'
MAX_PACKETS = 1024

os.makedirs(OUTPUT_DIR, exist_ok=True)

for fname in os.listdir(INPUT_DIR):
    if not fname.endswith('.nprint'):
        continue

    filepath = os.path.join(INPUT_DIR, fname)
    df = pd.read_csv(filepath)

    # Drop any non-numeric identifier column if present (e.g. a packet index column)
    df = df.select_dtypes(include=[np.number])

    # Restrict to the first 1024 packets
    df = df.iloc[:MAX_PACKETS]

    # nPrint uses -1 for "field not present" — map values to a fixed 0-255 grayscale range:
    #   -1 (absent)  -> 128 (mid-gray, visually distinct from both 0 and 1)
    #    0            -> 0   (black)
    #    1            -> 255 (white)
    arr = df.to_numpy()
    img_arr = np.zeros_like(arr, dtype=np.uint8)
    img_arr[arr == 0] = 0
    img_arr[arr == 1] = 255
    img_arr[arr == -1] = 128

    # Pad with a fill value if the trace has fewer than 1024 packets,
    # so every output image has a consistent height
    if img_arr.shape[0] < MAX_PACKETS:
        pad_rows = MAX_PACKETS - img_arr.shape[0]
        padding = np.full((pad_rows, img_arr.shape[1]), 64, dtype=np.uint8)  # distinct pad value
        img_arr = np.vstack([img_arr, padding])

    img = Image.fromarray(img_arr, mode='L')  # grayscale
    out_name = os.path.splitext(fname)[0] + '.png'
    img.save(os.path.join(OUTPUT_DIR, out_name))

print(f"Converted images saved to {OUTPUT_DIR}")

Converted images saved to data/nd_data/student_converted_images


In [10]:
with open('./scripts/nprint_to_png.py', encoding='utf-8') as f:
    content = f.read()

content = content.replace('.applymap(', '.map(')

with open('./scripts/nprint_to_png.py', 'w', encoding='utf-8') as f:
    f.write(content)

In [11]:
# The following is a pre-defined script used in NetDiffusion that will convert all of the provided real nprints into image representations. Run this code and observe the output
!python ./scripts/nprint_to_png.py -i data/nd_data/real_nprints/ -o data/nd_data/real_traffic_images

Processing amazon_1024.nprint
Processing amazon_1024_1.nprint
Processing amazon_1024_10.nprint
Processing amazon_1024_11.nprint
Processing amazon_1024_12.nprint
Processing amazon_1024_13.nprint
Processing amazon_1024_14.nprint
Processing amazon_1024_15.nprint
Processing amazon_1024_16.nprint
Processing amazon_1024_17.nprint
Processing amazon_1024_18.nprint
Processing amazon_1024_19.nprint
Processing amazon_1024_2.nprint
Processing amazon_1024_3.nprint
Processing amazon_1024_4.nprint
Processing amazon_1024_5.nprint
Processing amazon_1024_6.nprint
Processing amazon_1024_7.nprint
Processing amazon_1024_8.nprint
Processing amazon_1024_9.nprint
Processing facebook_1024.nprint
Processing facebook_1024_1.nprint
Processing facebook_1024_10.nprint
Processing facebook_1024_11.nprint
Processing facebook_1024_12.nprint
Processing facebook_1024_13.nprint
Processing facebook_1024_14.nprint
Processing facebook_1024_15.nprint
Processing facebook_1024_16.nprint
Processing facebook_1024_17.nprint
Proces

Q3: Now that you have seen how NetDiffusion converts nPrints into images, compare their method with the approach you designed in Q2. What are the advantages and disadvantages of each, especially in terms of what might help or hinder a vision model’s ability to learn?

**ANSWER**

The NetDiffusion method used three colors (Red, Green, Blue). NetDiffusion makes it easier for the model to learn, as the colors are more distinct and easier to read, especially when blurred. However, because it needs to store the information for three colors, it takes up more storage. 

The method designed in Q2 used white, black, and gray. This method has smaller file sizes, but once blurred, it makes it harder to distinguish the information.


---

## **Part 2 — Converting Generated Images Back Into Usable Format**

In the first part of the assignment, you explored how network traces in nPrint format can be transformed into image representations suitable for vision-based generative models. In Part 2, we focus on the reverse process: taking synthetic images produced by these models and converting them back into structured network representations.

This step is crucial because real-world applications do not operate on images—they require valid, interpretable packet traces that can be analyzed, replayed, or integrated into downstream tools.

---

### **Your Task for Part 2**

In this part of the assignment, you will:

1. **Convert generated images back into the original nPrint representation.**
   You will follow a scripted pipeline that translates pixel intensities and color channels back into binary header fields, reconstructing the packet-level structure of the trace.

2. **Apply essential post-processing techniques to correct errors introduced by diffusion models.**
   Generated images are rarely perfect—vision models may introduce color drift, pixel misalignment, noise, or structural artifacts.
   You will observe how heuristic correction, formatting enforcement, and reconstruction steps ensure that the converted nPrints become:

   * syntactically valid,
   * structurally consistent,
   * and replayable.

Across this section, your goal is to understand **why the reverse transformation is fragile**, which types of artifacts break reversibility, and how post-processing logic helps repair or compensate for generative errors.

For simplicity of this assignment, we have trainined and generated the images for you. If you have sufficient GPU access and want to try fine-tuning the model and generating the images yourself, feel free to take a look at the public repo (https://github.com/noise-lab/NetDiffusion).


Q4: We have taken the images converted by NetDiffusion and trained a LoRA-fine-tuned Stable Diffusion model (with ControlNet) to generate synthetic traffic images for you. These generated samples are stored in generated_traffic_images/.
Compare these generated images visually with the real images you saw earlier in real_traffic_images/.

Do you notice anything different between the real and generated traffic images? What immediately stands out as potentially problematic if we attempt to convert these generated images back into nPrint format? (Descriptive Only)

**The generated traffic images are blurrier than the real images. Pixels aren't just three colors and there's a lot of noise. This is going to be a problem when trying to convert it back to nPrint format because there are more than three bits to convert (-1/0/1).**

Q5: If you were to design a method to convert generated images back into nPrints, how would you do it? Explain your approach and describe how your method addresses the concerns you raised in Q4. (Descriptive only)

**I would make it so that each pixel of the image is adjusted to the closest to the original values. For example, if a pixel's value is 230, it is closest to 255, so it would turn into that value. Therefore, there aren't more than three bits to convert back and it will work. Then, I would have the method map each color to an nPrint value (0/1/-1). This method would prevent any confusion when converting blurred bits and make the generated images from the nPrints look good**.

Below are a set of pre-written scripts that perform the necessary post-generation augmentation and processing on the synthetic images you obtained from the diffusion model. These scripts handle tasks such as color normalization/augmentation and conversion from generated images back into nPrint format.
(The PCAP step is optional — you may run it if you are interested in observing or replaying the reconstructed traffic.)

In [15]:
# Step 1: Color Augmentation
!python ./scripts/color_processor.py \
  --input_dir="data/nd_data/generated_traffic_images" \
  --output_dir="data/nd_data/color_corrected_generated_traffic_images"

^C


In [22]:
# Step 2: Image-to-nPrint Conversion
!python ./scripts/image_to_nprint.py \
  --org_nprint ./scripts/column_example.nprint \
  --input_dir ./data/nd_data/color_corrected_generated_traffic_images \
  --output_dir ./data/nd_data/generated_nprint

Processing ./data/nd_data/color_corrected_generated_traffic_images\amazon_0.png with size 1088 x 1024
Saved ./data/nd_data/generated_nprint\amazon_0.nprint
Processing ./data/nd_data/color_corrected_generated_traffic_images\amazon_1.png with size 1088 x 1024
Saved ./data/nd_data/generated_nprint\amazon_1.nprint
Processing ./data/nd_data/color_corrected_generated_traffic_images\amazon_2.png with size 1088 x 1024
Saved ./data/nd_data/generated_nprint\amazon_2.nprint
Processing ./data/nd_data/color_corrected_generated_traffic_images\amazon_3.png with size 1088 x 1024
Saved ./data/nd_data/generated_nprint\amazon_3.nprint
Processing ./data/nd_data/color_corrected_generated_traffic_images\amazon_4.png with size 1088 x 1024
Saved ./data/nd_data/generated_nprint\amazon_4.nprint
Processing ./data/nd_data/color_corrected_generated_traffic_images\amazon_5.png with size 1088 x 1024
Saved ./data/nd_data/generated_nprint\amazon_5.nprint
Processing ./data/nd_data/color_corrected_generated_traffic_imag

Q6: You may now read through the provided scripts in color_processor.py (color augmentation / normalization) and image_to_nprint.py (image → nPrint reconstruction).
How do the post-processing methods implemented in these scripts compare to the approach you proposed in Q5?
Describe the pros and cons of both methods and highlight any differences in design philosophy, robustness, or assumptions.

**My suggested approach maps pixels based on how close they are to the values (red, green, blue) while the approach in the scripts uses a threshold method. In this case, my suggested approach doesn't rely on an assumption on what the blur is and is conceptually simpler. However, it doesn't account for noise or apply a smoothing filter. The approach in the scripts is faster than my proposed method, as it doesn't require a distance calculation. However, it also doesn't account for noise, and it has the added disadvantage of using hard threshold values that may result in incorrect pixels.**

**The main difference in design philosophy is the assumptions made. I assumed that the goal is to get the output with the least amount of error, therefore I went with the closest pixel value. The scripts assumption was that the specified ranges would give the most accurate color, however if a color doesn't match the thresholds, it can be misrepresented.**



---

# **Part 3 — Using Real and Synthetic nPrints for Application Classification**

In the previous parts, you learned how network traces can be converted between nPrint and image representations, generated using diffusion models, and reconstructed back into nPrint format.
Now, you will evaluate how useful these generated nPrints are for downstream **machine learning tasks**.

Each nPrint file—whether real or generated—is labeled with the **application** that produced the traffic (e.g., `amazon_1.nprint` means this sample came from Amazon traffic).
In this section, you will treat each **entire nPrint file as a single sample** and build a simple ML pipeline to classify application labels.

To simplify the task, you will restrict your model to use **only the first 3 packets** (3 rows) from each nPrint.
This mimics “early packet classification,” where only the beginning of a flow is available.

---


Q7:

You now have access to both `real_nprints/` and `generated_nprint/`.
Notice that in both directories, files are labeled using the application associated with that nPrint (e.g., `amazon_1.nprint`).
Treat each **nPrint file** as one sample.

**Design an ML pipeline that trains a model using *synthetic nPrints* (from `generated_nprint/`) and evaluates its performance on *real nPrints* (from `real_nprints/`) to predict the correct application label.**

Your pipeline should:

1. Use only the **first 3 packets (first 3 rows)** of each nPrint file as input features.
2. Train a classifier on real data.
3. Test the classifier on generated data.
4. Report how well the classifier performs.

We have already written the script to load the real and generated nprints into DataFrame for you.

In [23]:
import os
import glob
import numpy as np
import pandas as pd

# ---------------------------------------------------------------
# Safe conversion for nPrint cell
# ---------------------------------------------------------------
def safe_convert(x):
    if pd.isna(x):
        return 0
    x = str(x).strip()

    if x in ["0", "1", "-1"]:
        return int(x)

    if all(c in "01" for c in x) and len(x) <= 16:
        return int(x, 2)

    if x.lstrip("-").isdigit():
        return int(x)

    return 0


# ---------------------------------------------------------------
# Get original column names from a reference nPrint
# ---------------------------------------------------------------
def get_original_columns(example_path="data/nd_data/real_nprints"):
    first_file = glob.glob(os.path.join(example_path, "*.nprint"))[0]

    df = pd.read_csv(first_file)

    # Drop index column if present (like "Unnamed: 0")
    if df.columns[0].lower().startswith("unnamed"):
        df = df.drop(df.columns[0], axis=1)

    return list(df.columns)


# ---------------------------------------------------------------
# Load nPrint → first 3 rows → flatten with prefixed column names
# ---------------------------------------------------------------
def load_nprint_with_colnames(path, base_cols, num_rows=3):
    df = pd.read_csv(path, dtype=str, low_memory=False)

    # Drop "Unnamed: 0" if present
    if df.columns[0].lower().startswith("unnamed"):
        df = df.drop(df.columns[0], axis=1)

    df = df.iloc[:num_rows, :]            # first 3 packets  
    df = df.map(safe_convert)             # clean convert  

    # Build prefixed column names
    pkt_cols = []
    for pkt in range(1, num_rows + 1):
        pkt_cols.extend([f"pkt{pkt}_{c}" for c in base_cols])

    # Flatten 3×columns into 1 vector
    flat = df.values.flatten()

    return flat, pkt_cols


# ---------------------------------------------------------------
# Load entire directory into a DataFrame (with labels)
# ---------------------------------------------------------------
def load_directory_as_df(directory, base_cols):
    rows = []
    labels = []
    colnames_set = None

    for path in glob.glob(os.path.join(directory, "*.nprint")):
        label = os.path.basename(path).split("_")[0]

        flat, cn = load_nprint_with_colnames(path, base_cols)
        rows.append(flat)
        labels.append(label)

        if colnames_set is None:
            colnames_set = cn   # only set once

    df = pd.DataFrame(rows, columns=colnames_set)
    df["label"] = labels
    return df


# ---------------------------------------------------------------
# FINAL: Load real + synthetic DataFrames
# ---------------------------------------------------------------
base_cols = get_original_columns("data/nd_data/real_nprints")

df_synth = load_directory_as_df("data/nd_data/generated_nprint", base_cols)
df_real  = load_directory_as_df("data/nd_data/real_nprints", base_cols)

print("Synthetic DF:", df_synth.shape)
print(df_synth.head())

print("\nReal DF:", df_real.shape)
print(df_real.head())


Synthetic DF: (100, 3265)
   pkt1_ipv4_ver_0  pkt1_ipv4_ver_1  pkt1_ipv4_ver_2  pkt1_ipv4_ver_3  \
0                0                0                0                0   
1                1                0                0                0   
2                1                0                0                0   
3                1                0                0                0   
4                1                0                0                0   

   pkt1_ipv4_hl_0  pkt1_ipv4_hl_1  pkt1_ipv4_hl_2  pkt1_ipv4_hl_3  \
0               0               0               0               0   
1               0               0               0               0   
2               0               0               0               0   
3               0               0               0               0   
4               0               0               1               0   

   pkt1_ipv4_tos_0  pkt1_ipv4_tos_1  ...  pkt3_icmp_roh_23  pkt3_icmp_roh_24  \
0                0                0  ...

Code-->

In [24]:
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Features = all columns except the label
feature_cols = [c for c in df_synth.columns if c != "label"]

X_train = df_synth[feature_cols]
y_train = df_synth["label"]

X_test = df_real[feature_cols]
y_test = df_real["label"]

# Encode labels consistently across both sets
le = LabelEncoder()
le.fit(pd.concat([y_train, y_test]))
y_train_enc = le.transform(y_train)
y_test_enc = le.transform(y_test)

clf = RandomForestClassifier(random_state=42)
clf.fit(X_train, y_train_enc)

y_pred = clf.predict(X_test)

print("Accuracy:", accuracy_score(y_test_enc, y_pred))
print("F1 (macro):", f1_score(y_test_enc, y_pred, average='macro'))
print(classification_report(y_test_enc, y_pred, target_names=le.classes_))

Accuracy: 0.28
F1 (macro): 0.18868276999143271
              precision    recall  f1-score   support

      amazon       0.00      0.00      0.00        20
    facebook       0.46      0.30      0.36        20
   instagram       0.53      0.50      0.51        20
        meet       0.40      0.10      0.16        20
     netflix       0.50      0.05      0.09        20
       teams       0.27      0.85      0.41        20
      twitch       0.00      0.00      0.00        20
     twitter       0.21      1.00      0.34        20
     youtube       0.00      0.00      0.00        20
        zoom       0.00      0.00      0.00        20

    accuracy                           0.28       200
   macro avg       0.24      0.28      0.19       200
weighted avg       0.24      0.28      0.19       200



C:\Users\Yosan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\Yosan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\Yosan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\metrics\_clas

**Notes on Performance**
The model performed poorly, but this is likely because of errors being compounded and the small amount of packets used. If more packets were used, and there were less steps that the data went through before being processed here, then the accuracy would likely be higher. Additionally, the lack of noise prevention and smoothening probably also added to the models confusion and low performance. 